In [1]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm
import plotly.graph_objects as go


n_bins = 40

def exp( x , N, A , tau ):
    return N*expon.cdf(x , A , tau) 

def gauss( x , N , mu , sigma):
    return N*norm.cdf(x , mu , sigma)

def exp_gauss_cdf(N_exp , A , tau):
    def fixed_exp( x , N , mu , sigma):
        return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
    return fixed_exp
    
def gauss_exp_cdf(N_g , mu , sigma):
    def fixed_gauss( x , N , A , tau):
        return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
    return fixed_gauss

def exp_fit(cdf, N, A , tau, count , edges):
    
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , A = A , tau = tau )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n





In [ ]:
data = np.genfromtxt("Data/timestamp/2026-01-08.csv", delimiter=',')
print(len(data))

data = data[(data > 2e-6)]

print(len(data))

count, edges = np.histogram(data, bins=n_bins , density=False)

fig = go.Figure()

fig.add_trace(
    go.Bar(x=edges[:-1], y=count, name='Histogram', width=np.diff(edges))
)

fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

In [4]:
data = np.genfromtxt("Data/timestamp/2026-01-08.csv", delimiter=',')
print(len(data))

data = data[(data > 2e-6)]

print(len(data))

count, edges = np.histogram(data, bins=n_bins , density=False)

count -= 4000

n = exp_fit( exp ,1e5 ,1 , 2e-6 , count , edges)
display(n)







8636280
411199


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4.077e+06 (χ²/ndof = 110187.9)│              Nfcn = 926              │
│ EDM = nan (Goal: 0.0002)         │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N    │    1e5    │    nan    │            │            │         │         │       │
│ 1 │ A    │     1     │    nan    │            │            │         │         │       │
│ 2 │ tau  │   2e-6    │    nan    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────┐
│     │   N   A tau │
├─────┼─────────────┤
│   N │ nan nan nan │
│   A │ nan nan nan │
│ tau │ nan nan nan │
└─────┴─────────────┘

In [3]:
del_count = np.empty_like(count)

for i in range(len(count)):
    del_count[i] = np.round(float(count[i]) - float(exp( (edges[i+1] + edges[i])/2 , *n.values)))
    print(count[i],"\t", exp( (edges[i+1] + edges[i])/2 , *n.values) , float(count[i])/float(exp( (edges[i+1] + edges[i])/2 , *n.values)))
    if del_count[i]< 0 : del_count[i] = 0

print(del_count)

m = gauss_fit( gauss , 1e6 , 1.5e-6 , 0.3e-6 , del_count , edges)
display(m)

ZeroDivisionError: division by zero

In [ ]:
eg_cdf = exp_gauss_cdf( *n.values)

m = gauss_fit( eg_cdf , 1e6 , 1.5e-6 , 0.3e-6 , count , edges)
display(m)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.776e+04 (χ²/ndof = 265.1)│              Nfcn = 149              │
│ EDM = 0.000137 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N     │  63.2e3   │   0.9e3   │            │            │         │         │       │
│ 1 │ mu    │ 1.4299e-6 │ 0.0028e-6 │            │            │         │         │       │
│ 2 │ sigma │ 159.1e-9  │  1.8e-9   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────────────────────────────────────────┐
│       │                 N                mu             sigma │
├───────┼───────────────────────────────────────────────────────┤
│     N │          7.85e+05 -251.891893683e-9 601.3389620261e-9 │
│    mu │ -251.891893683e-9          8.06e-18           0.1e-18 │
│ sigma │ 601.3389620261e-9           0.1e-18          3.31e-18 │
└───────┴───────────────────────────────────────────────────────┘

In [ ]:
ge_cdf = gauss_exp_cdf( *m.values)

k = exp_fit( ge_cdf , 7.56e6 , -373e-9 , 0.857e-6 , count , edges)
display(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.573e+04 (χ²/ndof = 234.7)│              Nfcn = 76               │
│ EDM = 6.51e-08 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N    │   7.2e6   │   1.7e6   │            │            │         │         │       │
│ 1 │ A    │ -0.41e-6  │  0.21e-6  │            │            │         │         │       │
│ 2 │ tau  │ 888.1e-9  │  0.8e-9   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────────────────────────────┐
│     │                   N                   A                 tau │
├─────┼─────────────────────────────────────────────────────────────┤
│   N │            2.79e+12 -344.52311821969e-3 -3.5152993582255e-6 │
│   A │ -344.52311821969e-3            4.25e-14            -0.3e-18 │
│ tau │ -3.5152993582255e-6            -0.3e-18            5.93e-19 │
└─────┴─────────────────────────────────────────────────────────────┘

In [ ]:

data = np.genfromtxt("Data/timestamp/2026-01-08.csv", delimiter=',')
print(len(data))

data = data[(data > 7e-7)]

print(len(data))

count, edges = np.histogram(data, bins=n_bins , density=False)

for _ in range(1000):
    ge_cdf = gauss_exp_cdf( *m.values)
    k = exp_fit( ge_cdf , *k.values , count , edges)

    eg_cdf = exp_gauss_cdf( *k.values)
    m = gauss_fit( eg_cdf , *m.values, count , edges)

display(m)
display(k)

8636280
2071130


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.446e+04 (χ²/ndof = 215.8)│              Nfcn = 51               │
│ EDM = 3.13e-10 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N     │  109.4e3  │   1.0e3   │            │            │         │         │       │
│ 1 │ mu    │ 1.4060e-6 │ 0.0023e-6 │            │            │         │         │       │
│ 2 │ sigma │ 197.6e-9  │  1.8e-9   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────────────────────────────────────────┐
│       │                 N                mu             sigma │
├───────┼───────────────────────────────────────────────────────┤
│     N │          1.08e+06 -481.083772930e-9 872.4114391269e-9 │
│    mu │ -481.083772930e-9          5.29e-18          -0.7e-18 │
│ sigma │ 872.4114391269e-9          -0.7e-18          3.32e-18 │
└───────┴───────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.446e+04 (χ²/ndof = 215.8)│              Nfcn = 48               │
│ EDM = 9.65e-09 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N    │   7.1e6   │   2.8e6   │            │            │         │         │       │
│ 1 │ A    │ -0.42e-6  │  0.35e-6  │            │            │         │         │       │
│ 2 │ tau  │ 893.0e-9  │  0.8e-9   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬────────────────────────────────────────────────────────────────┐
│     │                    N                    A                  tau │
├─────┼────────────────────────────────────────────────────────────────┤
│   N │             7.75e+12  -975.13106022865e-3 -76.6413319081915e-6 │
│   A │  -975.13106022865e-3             1.23e-13              8.9e-18 │
│ tau │ -76.6413319081915e-6              8.9e-18             6.19e-19 │
└─────┴────────────────────────────────────────────────────────────────┘